In [203]:
# reload the modules
%load_ext autoreload
%autoreload 3

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [204]:
import numpy as np
from minitorch.tensor.tensor import Tensor
from minitorch.nn.layers import Linear, Sequential, Dropout
from minitorch.activations.activations import ReLU, GELU, Tanh, Sigmoid, Softmax
from minitorch.dataloaders.dataloader import DataLoader, TensorDataset
from minitorch.losses.losses import BinaryCrossEntropy
from minitorch.labs.loan_default.src.components.data_transformation import DataTransformation
from minitorch.labs.loan_default.src.pipeline.model_pipeline import LoanDefaultPredictor
from minitorch.optimizers.optim import AdamW, SGD
from minitorch.train.training import Trainer, CosineSchedule

Data Ingestion and Transformation


In [12]:
# Ingestion and transform the data
data_transformer = DataTransformation()
data_transformer.train_set = data_transformer.train_set[:1000]
data_transformer.test_set = data_transformer.test_set[:100]

train_arr, test_arr = data_transformer.initiate_data_transformation()

🧪 Ingesting data...
📥 Loading dataset from C:\Users\User\Desktop\Loan_Default.csv...
✅ Loaded dataset successfully.
🔀 Splitting dataset into training and testing sets...
✅ Split dataset successfully.
💾 Saving dataset to artifacts directory...
📁 Checking if directory exists ...
📁 Directory already exists at c:\Users\User\Desktop\babytorch\minitorch\labs\loan_default\artifacts.
✅ Data already ingested.
✅ Found existing training and testing datasets. Returning their paths.
🧪 Getting the preprocessing Object ...
✍️ Applying transformation to training datasets ...
✍️ Applying transformation to testing datasets ...
Applying transformation to the response variable ...
✅ Done with the transformatiom 
🪡🧵Concatenating the features and the targets back together ...
✅✅ Done with the concatination and the data transformatiom.
(118936, 49) (29734, 49)


In [242]:
# # get the features and target and convert them to tensors
features, target = train_arr[:, 1:-2], train_arr[:, -1]
features, target = Tensor(features, requires_grad=True), Tensor(target, requires_grad=False)

# # create the data loader
ds = TensorDataset(features, target)
train_loader = DataLoader(ds, batch_size=32, shuffle=True)

In [349]:
# create the model
from minitorch.losses.losses import BCEWithLogits

MAX_EPOCHS = 3
model = LoanDefaultPredictor(in_features=features.shape[1], out_features=1, drop_out_p=0.0)
loss_fn = BCEWithLogits()
# optimizer = SGD(model.parameters(), lr=0.03)
optimizer = AdamW(model.parameters(), lr=0.03, weight_decay=0.1)
# scheduler = CosineSchedule(max_lr=0.03, min_lr = 0.03, total_epochs=MAX_EPOCHS)
trainer = Trainer(model=model,
                loss_fn=loss_fn,
                optimizer=optimizer,
                # scheduler=scheduler,
                clip_gradients=True)



In [350]:
STEPS = 1
print()
print('Full model training...\n')
for epoch in range(MAX_EPOCHS):
    loss = trainer.train_epoch(train_loader)
        
    if (epoch + 1) % STEPS == 0:
        print(f'Epoch {epoch+1}/{MAX_EPOCHS} | Average Loss: {loss:.4f}')
        # print(f'Learning Rate at epoch {epoch+1}: {trainer.scheduler.get_lr(epoch):.6f}\n')
        
        for p in model.parameters():
            print(f'Parameter: {p.data.flatten()[:5]}')
        print('\n')


Full model training...

Epoch 1/3 | Average Loss: 694.2002
Parameter: [ 0.01990698  0.05697813  0.09189939 -0.05185256  0.26164505]
Parameter: [-0.02821119  0.05819686  0.11225384  0.22122964  0.02634322]
Parameter: [ 0.13537292 -0.07711895 -0.18396783  0.40201122  0.25334546]
Parameter: [ 0.00055585  0.0807635  -0.00289782 -0.19048123 -0.19604637]
Parameter: [ 0.1772797   0.09347265 -0.07457332  0.07026476  0.07802998]
Parameter: [ 0.05826735 -0.02924782  0.07103197  0.20091203  0.0295896 ]
Parameter: [-0.32172865 -0.09222799  0.04049677  0.05332916 -0.08881898]
Parameter: [ 0.07865122  0.31671807 -0.06700879 -0.13837302  0.00260517]
Parameter: [-0.08710787  0.09649301  0.22341293 -0.0577848   0.00744579]
Parameter: [ 0.11535861  0.30320516  0.03796534 -0.1857519  -0.08206466]
Parameter: [-0.08328783 -0.08732264  0.08219696  0.06932482 -0.18640004]
Parameter: [-0.38751158 -0.06897466 -0.21989793  0.00197939  0.12159608]
Parameter: [-0.05527194 -0.02561002 -0.14345199  0.00206342 -0.3

In [19]:
from minitorch.nn.layers import Linear

In [309]:
target.data

array([0., 0., 0., ..., 0., 0., 0.], shape=(118936,), dtype=float32)

In [320]:
features, targets = features[:20], target[:20]
in_features = features.shape[1]
out_features = 5
linear = Linear(in_features, 1, bias=False)
act = Sigmoid()
# out = act(
logits = linear(features)
# sigmoid_out = act(out)
# sigmoid_out
loss_fn = BCEWithLogits()
loss = loss_fn(logits, target)
loss.backward()